In [1]:
import sys
import os

# Set working directory to project root (/home/jovyan/work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

# Imports
from src.core.config import get_default_tickers
from src.data.ingestion import MarketDataIngestion
from src.data.spark_pipeline import get_spark_session, SparkTechnicalIndicators
from src.data.databricks_client import DatabricksClient

print("Imports successful!")

Imports successful!


In [2]:
tickers = get_default_tickers()
fetcher = MarketDataIngestion()
spark = get_spark_session()
indicator_calc = SparkTechnicalIndicators(spark)
db_client = DatabricksClient()

for ticker in tickers:
    print(f"Processing indicators for: {ticker}")
    # 1. Fetch OHLCV
    pandas_df = fetcher.fetch_daily_ohlcv(ticker)

    # 2. Spark Transformations
    spark_df = spark.createDataFrame(pandas_df)
    processed_df = indicator_calc.compute_indicators(spark_df)

    # 3. Store Data
    db_client.write_dataset(processed_df, table_name=f"{ticker.lower()}_indicators")

print("Phase 1 Execution Complete!")

Processing indicators for: AAPL
[2026-09-24 22:37:34] [INFO] [stonks_maker]: Fetching OHLCV for AAPL via yfinance...
[2026-09-24 22:37:36] [INFO] [stonks_maker]: Starting Spark indicator transformations...
[2026-09-24 22:37:36] [INFO] [stonks_maker]: Completed Spark indicator calculations.
[2026-09-24 22:37:36] [INFO] [stonks_maker]: Databricks credentials not configured. Saving locally to Parquet.
[2026-09-24 22:37:39] [INFO] [stonks_maker]: Successfully saved to /home/jovyan/work/data/processed/aapl_indicators
Processing indicators for: MSFT
[2026-09-24 22:37:39] [INFO] [stonks_maker]: Fetching OHLCV for MSFT via yfinance...
[2026-09-24 22:37:39] [INFO] [stonks_maker]: Starting Spark indicator transformations...
[2026-09-24 22:37:40] [INFO] [stonks_maker]: Completed Spark indicator calculations.
[2026-09-24 22:37:40] [INFO] [stonks_maker]: Databricks credentials not configured. Saving locally to Parquet.
[2026-09-24 22:37:40] [INFO] [stonks_maker]: Successfully saved to /home/jovyan/

In [3]:
# Verify saved Parquet/Delta dataset
output_df = db_client.read_dataset(spark, table_name="aapl_indicators")
output_df.show(5)

[2026-09-24 22:37:42] [INFO] [stonks_maker]: Reading dataset locally from /home/jovyan/work/data/processed/aapl_indicators
+----------+------+------------------+------------------+------------------+------------------+------------------+--------+------------------+------------------+------------------+------------------+------------------+
|      date|ticker|              open|              high|               low|             close|         adj_close|  volume|            sma_20|            sma_50|   bollinger_upper|   bollinger_lower|            rsi_14|
+----------+------+------------------+------------------+------------------+------------------+------------------+--------+------------------+------------------+------------------+------------------+------------------+
|2025-09-25|  AAPL|252.27808820484515|256.22352044118514| 250.7836088309123| 255.9246063232422| 255.9246063232422|55202100| 255.9246063232422| 255.9246063232422|              NULL|              NULL|               0.0|
|